In [3]:
import sys
sys.path.insert(0, "..")   # also look in the parent folder (the repo root)

import numpy as np
import helpers 
import pandas as pd
import preprocessing
import implementations

## loading datas

In [ ]:
## ON TIME

# data_path = '../data'

# x_train, x_test, y_train, train_ids, test_ids = helpers.load_csv_data(data_path)

# ## save the nparrays
# np.save("../data/x_train.npy", x_train)
# np.save("../data/x_test.npy", x_test)
# np.save("../data/y_train.npy", y_train)
# np.save("../data/train_ids.npy", train_ids)
# np.save("../data/test_ids.npy", test_ids)


In [ ]:
## load numpy arrays
x_train = np.load("../data/x_train.npy")
x_test = np.load("../data/x_test.npy")
y_train = np.load("../data/y_train.npy")
train_ids = np.load("../data/train_ids.npy")
test_ids = np.load("../data/test_ids.npy")

## convert the labels of y_train to 0,1
y = (y_train == 1).astype(int)

In [41]:
## dim check
print(x_train.shape, y_train.shape, train_ids.shape, x_test.shape, test_ids.shape)
print(y, np.mean(y))


(328135, 321) (328135,) (328135,) (109379, 321) (109379,)
[0 0 0 ... 0 0 0] 0.08830207079403295


## preprocessing - test of antho pipeline : 

In [21]:
# load the features name 
names = preprocessing.load_feature_names(data_path)

# find the preprocessing statistic :
params = preprocessing.fit_preprocessing(x_train, names)

# transform the original dataset in a usable dataset
tx_wob = preprocessing.transform(x_train, params)

# add a biais : 
ones = np.ones((tx_wob.shape[0], 1)) # give the size ()
tx = np.concatenate([ones, tx_wob], axis=1)

In [34]:
## checks 

print(tx, tx.shape, np.mean(tx), np.std(tx))
print(np.mean(tx_wob), np.std(tx_wob))

col_mean = tx_wob.mean(axis=0)    # checked column by column 
print(col_mean.shape)
col_std  = tx_wob.std(axis=0)
print(np.abs(col_mean).max())        # should be ~0 (e.g. 1e-12)
print(col_std.min(), col_std.max())  # should both be ~1
print(np.all(tx[:, 0] == 1))         # bias column is all 1s → True


print(np.isnan(tx).sum())


[[ 1.         -0.0916959  -0.1348452  ... -0.29674928  0.65990453
   0.59027004]
 [ 1.         -0.0916959  -0.1348452  ... -0.29674928  0.65990453
   0.59027004]
 [ 1.         -0.0916959  -0.1348452  ...  3.36984807 -1.51537071
   0.59027004]
 ...
 [ 1.         -0.0916959  -0.1348452  ...  3.36984807 -1.51537071
   0.59027004]
 [ 1.         -0.0916959  -0.1348452  ... -0.29674928  0.65990453
   0.59027004]
 [ 1.         -0.0916959  -0.1348452  ... -0.29674928  0.65990453
   0.59027004]] (328135, 349) 0.0028653295129141268 0.9999958949348441
2.020169593538713e-14 0.9999999999998783
(348,)
1.0033101145940887e-11
0.9999999999925855 1.000000000005876
True
0


In [39]:
del names, params, tx_wob, tx 

## metric fun to test - F1 score and treshold



In [ ]:
# F1 score (y_true, y_pred) :


def f1_score(y_true, y_pred):

    TP = np.sum(y_true == 1 & y_pred == 1)
    FP = np.sum(y_true == 0 & y_pred == 1)
    FN = np.sum(y_true == 1 & y_pred == 0)
    TN = np.sum(y_true == 0 & y_pred == 0)

    precision = TP / (TP + FP)
    recall = TP / (TP + FN)

    f1 = 2 * (precision*recall) / (precision+recall)

    return f1




def best_threshold(y_true, scores):
    'takes the rough scores probabilities of log reg in input'
    'goal : find the best treshold to assign a 0 or a 1'
    ''

    thresholds = np.linspace(0, 1, 100) # difefrent treshold between 0 and 1

    f1_scores = []

    for t in thresholds:
        y_pred = (scores>=t).astype(int)
        f1 = f1_score(y_true, y_pred)

        f1_scores.append(f1)

    best_idx = np.argmax(f1_scores)

    best_t = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    return best_t, best_f1






## first model : 80/20 split (no cross-val)

In [ ]:
# 1 - from the loaded data - take a 50k rows subsample 

# 2 - from the subsample split the data - 80% training / 20% validation (preprocess after and test the model on it)

# 3 - preprocess the training data - fit + transform + biais

# 3 - preprocess the validation data with the fit of trai data - transform + biais 

# 4 - train the model using tx_train and y_train

# 5 - prediction on validation data (we ve just trained) + analyse 

In [ ]:
# 1 - from the loaded data - take a 50k rows subsample 
size = 50000

def subsample(x_train, y_train, size, rng):
    """"return 'size' random rows of x and the matching labels of y (track the idx)"""

    idx = rng.choice(len(y), size=size, replace=False)  # `size` distinct row numbers in 0..N-1 - no replace avoid duplication

    return x_train[idx], y_train[idx]


rng = np.random.default_rng(42)

x_sub, y_sub = subsample(x_train, y, 50_000, rng)
print(x_sub.shape, y_sub.shape, y_sub.mean())   # (50000, 321) (50000,) ~0.088

(50000, 321) (50000,) 0.0894


In [ ]:
# 2 - from the subsample split the data - 80% training / 20% validation (preprocess after and test the model on it)
ratio = 0.8

def split_data(x, y, ratio, rng):
    """return training and validation datas"""

    cut = int(ratio*len(y))

    idx = rng.permutation(len(y))
    idx_tr = idx[:cut]
    idx_val = idx[cut:]

    x_tr, y_tr   = x[idx_tr], y[idx_tr]
    x_val, y_val = x[idx_val], y[idx_val]

    return x_tr, y_tr, x_val, y_val

In [ ]:
# 3 - preprocess the training data - fit + transform + biais

# load the features name 
names = preprocessing.load_feature_names(data_path)

def train_preprocessing(x_tr, y_tr, names):
    """
    
    """
    
    # find the preprocessing statistic of x_tr:
    params_tr = preprocessing.fit_preprocessing(x_tr, names)
    
    # transform using the params_tr
    tx_wob = preprocessing.transform(x_tr, params_tr)

    ## add biais 
    tx = preprocessing.add_bias(tx_wob)

    ## y_tr in format
    y = (y_tr == 1).astype(int)

    return tx, y, params_tr


# 4 - preprocess the validation data with the fit of trai data - transform + biais 

def val_preprocessing(x_val, y_val, params_tr):
    """
    
    """
    
    # transform using params_tr on validation datas
    tx_wob = preprocessing.transform(x_val, params_tr)

    ## add biais 
    tx_val = preprocessing.add_bias(tx_wob)

    ## y_tr in format
    y = (y_val == 1).astype(int)

    return tx_val, y

In [44]:
## 3 - preprocessing --- final version :

# load the features name 
names = preprocessing.load_feature_names(data_path)

# y in format 
y = (y_train == 1).astype(int)

def prepare_features(x_tr, x_val, names):
    """Fit preprocessing on x_tr only, transform both, add bias."""
    params = preprocessing.fit_preprocessing(x_tr, names)
    tx_tr  = preprocessing.add_bias(preprocessing.transform(x_tr, params))
    tx_val = preprocessing.add_bias(preprocessing.transform(x_val, params))
    return tx_tr, tx_val

In [45]:
# 4 - train the model using tx_train and y_train
"""choose the model"""

## parameters
lambda_ = 1e4     # really small (almost no reg) -- tune it later
initial_w = np.zeros((tx_tr.shape[1])) # standards
max_iters = 500
gamma = 0.1 # step size


## training : 

w, loss = implementations.reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)



NameError: name 'tx_tr' is not defined

In [ ]:
# 5 - prediction on validation data (we ve just trained) + analyse 

y_pred = tx_val @ w

f1 = f1_score(y_val, y_pred)
print(f1)

In [8]:
"""ALL FUNC"""

def f1_score(y_true, y_pred):

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    if TP == 0:
        return 0.0

    precision = TP / (TP + FP)
    recall = TP / (TP + FN)
    f1 = 2 * (precision*recall) / (precision+recall)

    return f1


def best_threshold(y_true, scores):
    'takes the rough scores probabilities of log reg in input'
    'goal : find the best treshold to assign a 0 or a 1'
    ''

    thresholds = np.linspace(0, 1, 100) # difefrent treshold between 0 and 1

    f1_scores = []

    for t in thresholds:
        y_pred = (scores>=t).astype(int)
        f1 = f1_score(y_true, y_pred)

        f1_scores.append(f1)

    best_idx = np.argmax(f1_scores)

    best_t = thresholds[best_idx]
    best_f1 = f1_scores[best_idx]

    return best_t, best_f1


def subsample(x_train, y_train, size, rng):
    """"return 'size' random rows of x and the matching labels of y (track the idx)"""

    idx = rng.choice(len(y_train), size=size, replace=False)  # `size` distinct row numbers in 0..N-1 - no replace avoid duplication

    return x_train[idx], y_train[idx]


def split_data(x, y, ratio, rng):
    """return training and validation datas"""

    cut = int(ratio*len(y))

    idx = rng.permutation(len(y))
    idx_tr = idx[:cut]
    idx_val = idx[cut:]

    x_tr, y_tr   = x[idx_tr], y[idx_tr]
    x_val, y_val = x[idx_val], y[idx_val]

    return x_tr, y_tr, x_val, y_val


def prepare_features(x_tr, x_val, names):
    """Fit preprocessing on x_tr only, transform both, add bias."""
    params = preprocessing.fit_preprocessing(x_tr, names)
    tx_tr  = preprocessing.add_bias(preprocessing.transform(x_tr, params))
    tx_val = preprocessing.add_bias(preprocessing.transform(x_val, params))
    return tx_tr, tx_val



In [9]:
"""IMPLEMENTATION"""

rng = np.random.default_rng(42)

# 0 - load the datas
x_train = np.load("../data/x_train.npy")
y_train = np.load("../data/y_train.npy")
y = (y_train == 1).astype(int)   # y in format 


# 1 - from the loaded data - take a 50k rows subsample 
x_sub, y_sub = subsample(x_train, y, 50_000, rng)


# 2 - from the subsample split the data - 80% training / 20% validation (preprocess after and test the model on it)
x_tr, y_tr, x_val, y_val = split_data(x_sub, y_sub, 0.8, rng)


# 3 - preprocess the datas
names = preprocessing.load_feature_names("../data")   # load the features name 
tx_tr, tx_val = prepare_features(x_tr, x_val, names)


# 4 - train the model using tx_train and y_train
lambda_ = 1e-4     # really small (almost no reg) -- tune it later
initial_w = np.zeros((tx_tr.shape[1])) # standards
max_iters = 500
gamma = 0.1 # step size

w, loss = implementations.reg_logistic_regression(y_tr, tx_tr, lambda_, initial_w, max_iters, gamma)


# 5 - prediction on validation data (we ve just trained) + analyse 
y_pred = tx_val @ w
f1 = f1_score(y_val, y_pred)
print(f1)

0.0


In [10]:
scores = 1 / (1 + np.exp(-(tx_val @ w)))
print("loss:", loss)
print("scores min/mean/max:", scores.min(), scores.mean(), scores.max())
print("y_val:", np.unique(y_val, return_counts=True))
print("best:", best_threshold(y_val, scores))

loss: 0.22371707816506103
scores min/mean/max: 0.002882718771141559 0.0997651298068216 0.9240874752910455
y_val: (array([0, 1]), array([9104,  896]))
best: (np.float64(0.25252525252525254), np.float64(0.4543160020191822))
